In [6]:
import os
import random
from PIL import Image, ImageEnhance
import math
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, filename='process.log', format='%(asctime)s - %(message)s')


In [7]:
input_directory = r'C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\test_data\rawimg\Non_Autistic'
output_directory = r'C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\test_data\prepImg\step2\Non_Autistic'

input_files = os.listdir(input_directory)

if not os.path.exists(output_directory):
    os.makedirs(output_directory)

In [8]:
# Define the augmentations
def random_rotate(img):
    angle = random.randint(-30, 30)  # Random angle between -30 and 30 degrees
    return img.rotate(angle)

def random_flip(img):
    return img.transpose(Image.FLIP_LEFT_RIGHT)

def random_zoom(img):
    width, height = img.size
    factor = random.uniform(1.1, 1.5)  # Random zoom factor between 1.1 and 1.5
    new_size = (int(width * factor), int(height * factor))
    img_resized = img.resize(new_size, Image.Resampling.LANCZOS)  # Updated line
    left = (new_size[0] - width) // 2
    top = (new_size[1] - height) // 2
    right = (new_size[0] + width) // 2
    bottom = (new_size[1] + height) // 2
    img_cropped = img_resized.crop((left, top, right, bottom))
    return img_cropped

def random_shift(img):
    width, height = img.size
    shift_x = random.randint(-30, 30)  # Random horizontal shift
    shift_y = random.randint(-30, 30)  # Random vertical shift
    img_shifted = img.transform(img.size, Image.AFFINE, (1, 0, shift_x, 0, 1, shift_y))
    return img_shifted

def random_brightness(img):
    enhancer = ImageEnhance.Brightness(img)
    factor = random.uniform(0.7, 1.3)  # Random brightness factor between 0.7 and 1.3
    return enhancer.enhance(factor)

def random_contrast(img):
    enhancer = ImageEnhance.Contrast(img)
    factor = random.uniform(0.7, 1.3)  # Random contrast factor between 0.7 and 1.3
    return enhancer.enhance(factor)


In [9]:
# List of augmentations
augmentations = [random_rotate, random_flip, random_zoom, random_shift, random_brightness, random_contrast]

# Maximum number of times each augmentation should be applied
max_augmentations_per_type = 300
augmentation_counter = {aug: 0 for aug in augmentations}  # Track how many times each augmentation is applied

# List of files selected for augmentation
num_images_to_augment = 500
selected_files = random.sample(input_files, num_images_to_augment)

In [11]:
# Process each image and apply augmentation
for file in selected_files:
    file_path = os.path.join(input_directory, file)
    if file.lower().endswith(('.jpg', '.jpeg', '.png')):  # Process only image files
        try:
            # Open the image
            img = Image.open(file_path)

            # Track which augmentations have been used for the current image
            used_augmentation = None

            # Randomly apply augmentations and ensure no repetition
            while used_augmentation is None:
                aug = random.choice(augmentations)
                # Check if the augmentation has been applied less than the allowed number of times
                if augmentation_counter[aug] < max_augmentations_per_type:
                    augmented_img = aug(img)  # Apply the augmentation
                    used_augmentation = aug  # Mark the augmentation as used

                    # Save the augmented image
                    output_file = os.path.join(output_directory, f"aug_{file}")
                    augmented_img.save(output_file)

                    # Increment the count for the applied augmentation
                    augmentation_counter[aug] += 1
                    logging.info(f"Saved augmented image: {output_file}")
                else:
                    logging.warning(f"Skipping augmentation {aug} for {file} due to max count reached.")

        except Exception as e:
            logging.error(f"Error processing {file}: {e}")

logging.info("Processing complete.")